# GridLock Train-Only Modeling and Submission

This notebook trains and tunes models using `train.csv` only, then formats `test.csv` predictions according to `sample_submission.csv`.

Data policy: `training.csv` is intentionally not used in this notebook.


In [ ]:
# Run this cell only if your notebook environment is missing the required packages.
# %pip install optuna scikit-learn lightgbm xgboost joblib


In [ ]:
from __future__ import annotations

import argparse
import json
import math
import random
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
VENDOR = ROOT / ".vendor"
if VENDOR.exists():
    sys.path.insert(0, str(VENDOR))

import joblib
import numpy as np
import optuna
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from xgboost import XGBRegressor

try:
    from lightgbm import LGBMRegressor, early_stopping, log_evaluation
except Exception:
    LGBMRegressor = None
    early_stopping = None
    log_evaluation = None


warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
OUT = ROOT / "model_outputs" / "train_only"
OUT.mkdir(parents=True, exist_ok=True)

BASE32 = "0123456789bcdefghjkmnpqrstuvwxyz"
BASE32_MAP = {c: i for i, c in enumerate(BASE32)}


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)


def parse_minutes(value: object) -> float:
    if pd.isna(value):
        return np.nan
    hour, minute = str(value).split(":")
    return int(hour) * 60 + int(minute)


def geohash_decode(geohash: object) -> tuple[float, float, float, float, float]:
    if pd.isna(geohash):
        return np.nan, np.nan, np.nan, np.nan, np.nan

    lat_interval = [-90.0, 90.0]
    lon_interval = [-180.0, 180.0]
    is_even = True
    bit_value = 0

    for char in str(geohash):
        cd = BASE32_MAP.get(char)
        if cd is None:
            return np.nan, np.nan, np.nan, np.nan, np.nan
        bit_value = bit_value * 32 + cd
        for mask in [16, 8, 4, 2, 1]:
            if is_even:
                mid = (lon_interval[0] + lon_interval[1]) / 2
                if cd & mask:
                    lon_interval[0] = mid
                else:
                    lon_interval[1] = mid
            else:
                mid = (lat_interval[0] + lat_interval[1]) / 2
                if cd & mask:
                    lat_interval[0] = mid
                else:
                    lat_interval[1] = mid
            is_even = not is_even

    lat = (lat_interval[0] + lat_interval[1]) / 2
    lon = (lon_interval[0] + lon_interval[1]) / 2
    return lat, lon, lat_interval[1] - lat_interval[0], lon_interval[1] - lon_interval[0], float(bit_value)


def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    train = pd.read_csv(ROOT / "train.csv")
    test = pd.read_csv(ROOT / "test.csv")
    return train, test


def clean_frame(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ["geohash", "timestamp", "RoadType", "LargeVehicles", "Landmarks", "Weather"]:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()
    return df


def add_base_features(df: pd.DataFrame) -> pd.DataFrame:
    df = clean_frame(df)

    df["timestamp_minutes"] = df["timestamp"].map(parse_minutes)
    df["hour"] = np.floor(df["timestamp_minutes"] / 60)
    df["minute"] = df["timestamp_minutes"] % 60
    df["slot"] = np.floor(df["timestamp_minutes"] / 15)
    df["day_mod7"] = df["day"] % 7
    df["is_day49"] = (df["day"] == 49).astype(int)
    df["is_peak_commute"] = df["hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)
    df["is_late_night"] = df["hour"].isin([0, 1, 2, 3, 4]).astype(int)
    df["is_business_hour"] = df["hour"].between(9, 18).astype(int)

    minutes = df["timestamp_minutes"].fillna(0)
    df["time_sin"] = np.sin(2 * np.pi * minutes / 1440)
    df["time_cos"] = np.cos(2 * np.pi * minutes / 1440)
    df["slot_sin"] = np.sin(2 * np.pi * df["slot"].fillna(0) / 96)
    df["slot_cos"] = np.cos(2 * np.pi * df["slot"].fillna(0) / 96)

    df["Temperature_missing"] = df["Temperature"].isna().astype(int)
    df["Temperature_clip"] = df["Temperature"].clip(-20, 60)
    df["Temperature_sq"] = df["Temperature_clip"] ** 2
    df["Temperature_x_hour"] = df["Temperature_clip"] * df["hour"]
    df["lanes_x_hour"] = df["NumberofLanes"] * df["hour"]
    df["lanes_x_peak"] = df["NumberofLanes"] * df["is_peak_commute"]

    for col in ["RoadType", "LargeVehicles", "Landmarks", "Weather"]:
        df[f"{col}_missing"] = df[col].isna().astype(int)
        df[col] = df[col].fillna("Missing")

    df["large_allowed"] = (df["LargeVehicles"] == "Allowed").astype(int)
    df["has_landmark"] = (df["Landmarks"] == "Yes").astype(int)
    df["road_lanes"] = df["RoadType"].astype(str) + "_lanes_" + df["NumberofLanes"].astype(str)
    df["weather_road"] = df["Weather"].astype(str) + "_" + df["RoadType"].astype(str)

    df["geohash_prefix3"] = df["geohash"].astype(str).str[:3]
    df["geohash_prefix4"] = df["geohash"].astype(str).str[:4]
    df["geohash_prefix5"] = df["geohash"].astype(str).str[:5]

    decoded = {gh: geohash_decode(gh) for gh in df["geohash"].dropna().unique()}
    decoded_df = pd.DataFrame.from_dict(
        decoded,
        orient="index",
        columns=["geo_lat", "geo_lon", "geo_lat_err", "geo_lon_err", "geohash_int"],
    )
    decoded_df.index.name = "geohash"
    df = df.merge(decoded_df.reset_index(), on="geohash", how="left")
    df["geo_lat_x_lon"] = df["geo_lat"] * df["geo_lon"]

    for i in range(6):
        df[f"geohash_char{i}"] = df["geohash"].astype(str).str[i].map(BASE32_MAP).fillna(-1)

    return df


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))


def clip_pred(pred: np.ndarray) -> np.ndarray:
    return np.clip(np.asarray(pred, dtype=float), 0.0, 1.0)


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    pred = clip_pred(y_pred)
    return {
        "rmse": rmse(y_true, pred),
        "mae": float(mean_absolute_error(y_true, pred)),
        "r2": float(r2_score(y_true, pred)),
    }


@dataclass
class FeatureArtifacts:
    feature_columns: list[str]
    medians: pd.Series
    one_hot_columns: list[str]
    target_maps: dict[str, dict[Any, float]]
    target_global_mean: float
    freq_maps: dict[str, dict[Any, float]]
    count_maps: dict[str, dict[Any, int]]


class FeatureBuilder:
    def __init__(self) -> None:
        self.low_card_cols = [
            "RoadType",
            "NumberofLanes",
            "LargeVehicles",
            "Landmarks",
            "Weather",
            "road_lanes",
            "weather_road",
            "hour",
            "minute",
            "day_mod7",
        ]
        self.freq_cols = [
            "geohash",
            "geohash_prefix3",
            "geohash_prefix4",
            "geohash_prefix5",
            "timestamp",
            "RoadType",
            "Weather",
            "road_lanes",
        ]
        self.target_cols = [
            "geohash",
            "geohash_prefix3",
            "geohash_prefix4",
            "geohash_prefix5",
            "timestamp",
            "hour",
            "slot",
            "RoadType",
            "Weather",
            "LargeVehicles",
            "Landmarks",
            "road_lanes",
            "weather_road",
            "geo_time_key",
            "prefix5_time_key",
            "geo_hour_key",
            "road_weather_time_key",
        ]
        self.numeric_base_cols = [
            "day",
            "timestamp_minutes",
            "hour",
            "minute",
            "slot",
            "day_mod7",
            "is_day49",
            "is_peak_commute",
            "is_late_night",
            "is_business_hour",
            "NumberofLanes",
            "Temperature",
            "Temperature_missing",
            "Temperature_clip",
            "Temperature_sq",
            "Temperature_x_hour",
            "lanes_x_hour",
            "lanes_x_peak",
            "RoadType_missing",
            "LargeVehicles_missing",
            "Landmarks_missing",
            "Weather_missing",
            "large_allowed",
            "has_landmark",
            "time_sin",
            "time_cos",
            "slot_sin",
            "slot_cos",
            "geo_lat",
            "geo_lon",
            "geo_lat_err",
            "geo_lon_err",
            "geohash_int",
            "geo_lat_x_lon",
            "geohash_char0",
            "geohash_char1",
            "geohash_char2",
            "geohash_char3",
            "geohash_char4",
            "geohash_char5",
            "prev_day_same_geo_time_demand",
            "prev_day_geo_mean",
            "prev_day_timestamp_mean",
            "prev_day_geo_hour_mean",
            "prev_day_prefix5_time_mean",
        ]

    def _add_composite_keys(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["geo_time_key"] = df["geohash"].astype(str) + "_" + df["timestamp"].astype(str)
        df["prefix5_time_key"] = df["geohash_prefix5"].astype(str) + "_" + df["timestamp"].astype(str)
        df["geo_hour_key"] = df["geohash"].astype(str) + "_h" + df["hour"].astype(str)
        df["road_weather_time_key"] = (
            df["RoadType"].astype(str) + "_" + df["Weather"].astype(str) + "_" + df["hour"].astype(str)
        )
        return df

    def _add_lag_features(self, df: pd.DataFrame, source: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
        df = df.copy()
        source = source.copy()
        source["_target"] = y.to_numpy()

        exact = source[["geohash", "day", "timestamp", "_target"]].copy()
        exact["day"] = exact["day"] + 1
        exact = exact.rename(columns={"_target": "prev_day_same_geo_time_demand"})
        df = df.merge(exact, on=["geohash", "day", "timestamp"], how="left")

        geo = source.groupby(["geohash", "day"])["_target"].mean().reset_index()
        geo["day"] = geo["day"] + 1
        geo = geo.rename(columns={"_target": "prev_day_geo_mean"})
        df = df.merge(geo, on=["geohash", "day"], how="left")

        ts = source.groupby(["timestamp", "day"])["_target"].mean().reset_index()
        ts["day"] = ts["day"] + 1
        ts = ts.rename(columns={"_target": "prev_day_timestamp_mean"})
        df = df.merge(ts, on=["timestamp", "day"], how="left")

        source["hour"] = source["timestamp"].map(parse_minutes).floordiv(60)
        geo_hour = source.groupby(["geohash", "day", "hour"])["_target"].mean().reset_index()
        geo_hour["day"] = geo_hour["day"] + 1
        geo_hour = geo_hour.rename(columns={"_target": "prev_day_geo_hour_mean"})
        df = df.merge(geo_hour, on=["geohash", "day", "hour"], how="left")

        source["geohash_prefix5"] = source["geohash"].astype(str).str[:5]
        p5_time = source.groupby(["geohash_prefix5", "day", "timestamp"])["_target"].mean().reset_index()
        p5_time["day"] = p5_time["day"] + 1
        p5_time = p5_time.rename(columns={"_target": "prev_day_prefix5_time_mean"})
        df = df.merge(p5_time, on=["geohash_prefix5", "day", "timestamp"], how="left")

        return df

    def _fit_one_hot(self, fit_base: pd.DataFrame) -> list[str]:
        dummies = pd.get_dummies(fit_base[self.low_card_cols].astype("string"), prefix=self.low_card_cols, dtype=np.float32)
        return dummies.columns.tolist()

    def _transform_one_hot(self, df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
        dummies = pd.get_dummies(df[self.low_card_cols].astype("string"), prefix=self.low_card_cols, dtype=np.float32)
        return dummies.reindex(columns=columns, fill_value=0)

    def _fit_frequency_maps(self, fit_base: pd.DataFrame) -> tuple[dict[str, dict[Any, float]], dict[str, dict[Any, int]]]:
        freq_maps: dict[str, dict[Any, float]] = {}
        count_maps: dict[str, dict[Any, int]] = {}
        n = max(len(fit_base), 1)
        for col in self.freq_cols:
            counts = fit_base[col].astype("string").fillna("<NA>").value_counts()
            count_maps[col] = counts.to_dict()
            freq_maps[col] = (counts / n).to_dict()
        return freq_maps, count_maps

    def _add_frequency_features(
        self,
        df: pd.DataFrame,
        freq_maps: dict[str, dict[Any, float]],
        count_maps: dict[str, dict[Any, int]],
    ) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        for col in self.freq_cols:
            values = df[col].astype("string").fillna("<NA>")
            out[f"{col}_freq"] = values.map(freq_maps[col]).fillna(0.0).astype(np.float32)
            out[f"{col}_count"] = values.map(count_maps[col]).fillna(0.0).astype(np.float32)
        return out

    def _smoothed_map(self, values: pd.Series, y: pd.Series, smoothing: float, global_mean: float) -> dict[Any, float]:
        stats = pd.DataFrame({"value": values.astype("string").fillna("<NA>"), "target": y.to_numpy()})
        grouped = stats.groupby("value")["target"].agg(["mean", "count"])
        smooth = (grouped["mean"] * grouped["count"] + global_mean * smoothing) / (grouped["count"] + smoothing)
        return smooth.to_dict()

    def _target_encoding_oof(
        self,
        fit_base: pd.DataFrame,
        y: pd.Series,
        smoothing: float = 20.0,
        n_splits: int = 5,
    ) -> tuple[pd.DataFrame, dict[str, dict[Any, float]], float]:
        y = y.reset_index(drop=True)
        base = fit_base.reset_index(drop=True)
        global_mean = float(y.mean())
        enc = pd.DataFrame(index=base.index)
        maps: dict[str, dict[Any, float]] = {}
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        for col in self.target_cols:
            col_values = base[col].astype("string").fillna("<NA>")
            oof = np.full(len(base), global_mean, dtype=np.float32)
            for tr_idx, va_idx in kf.split(base):
                mapping = self._smoothed_map(col_values.iloc[tr_idx], y.iloc[tr_idx], smoothing, global_mean)
                oof[va_idx] = col_values.iloc[va_idx].map(mapping).fillna(global_mean).astype(np.float32)
            enc[f"{col}_te"] = oof
            maps[col] = self._smoothed_map(col_values, y, smoothing, global_mean)
        return enc, maps, global_mean

    def _target_encoding_apply(
        self,
        df: pd.DataFrame,
        maps: dict[str, dict[Any, float]],
        global_mean: float,
    ) -> pd.DataFrame:
        enc = pd.DataFrame(index=df.index)
        for col in self.target_cols:
            values = df[col].astype("string").fillna("<NA>")
            enc[f"{col}_te"] = values.map(maps[col]).fillna(global_mean).astype(np.float32)
        return enc

    def build(
        self,
        fit_df: pd.DataFrame,
        fit_y: pd.Series,
        apply_df: pd.DataFrame | None = None,
    ) -> tuple[pd.DataFrame, pd.DataFrame | None, FeatureArtifacts]:
        fit_base = self._add_composite_keys(add_base_features(fit_df))
        fit_base = self._add_lag_features(fit_base, fit_df, fit_y)

        apply_base = None
        if apply_df is not None:
            apply_base = self._add_composite_keys(add_base_features(apply_df))
            apply_base = self._add_lag_features(apply_base, fit_df, fit_y)

        one_hot_cols = self._fit_one_hot(fit_base)
        freq_maps, count_maps = self._fit_frequency_maps(fit_base)
        fit_te, target_maps, global_mean = self._target_encoding_oof(fit_base, fit_y)

        fit_parts = [
            fit_base[self.numeric_base_cols].copy(),
            self._transform_one_hot(fit_base, one_hot_cols),
            self._add_frequency_features(fit_base, freq_maps, count_maps),
            fit_te,
        ]
        X_fit = pd.concat(fit_parts, axis=1)

        X_apply = None
        if apply_base is not None:
            apply_parts = [
                apply_base[self.numeric_base_cols].copy(),
                self._transform_one_hot(apply_base, one_hot_cols),
                self._add_frequency_features(apply_base, freq_maps, count_maps),
                self._target_encoding_apply(apply_base, target_maps, global_mean),
            ]
            X_apply = pd.concat(apply_parts, axis=1)

        X_fit = self._finalize_numeric(X_fit)
        medians = X_fit.median(numeric_only=True).fillna(0.0)
        X_fit = X_fit.fillna(medians).astype(np.float32)
        if X_apply is not None:
            X_apply = self._finalize_numeric(X_apply)
            X_apply = X_apply.reindex(columns=X_fit.columns, fill_value=0.0).fillna(medians).astype(np.float32)

        artifacts = FeatureArtifacts(
            feature_columns=X_fit.columns.tolist(),
            medians=medians,
            one_hot_columns=one_hot_cols,
            target_maps=target_maps,
            target_global_mean=global_mean,
            freq_maps=freq_maps,
            count_maps=count_maps,
        )
        return X_fit, X_apply, artifacts

    def _finalize_numeric(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.replace([np.inf, -np.inf], np.nan)
        for col in X.columns:
            X[col] = pd.to_numeric(X[col], errors="coerce")
        return X


def make_validation_split(train: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    enriched = train.copy()
    enriched["timestamp_minutes"] = enriched["timestamp"].map(parse_minutes)
    val_mask = (enriched["day"] == 49) & (enriched["timestamp_minutes"] >= 90)
    if val_mask.sum() < 1000:
        val_mask = enriched["day"] == enriched["day"].max()
    return train.loc[~val_mask].copy(), train.loc[val_mask].copy()


def get_model(name: str, params: dict[str, Any]) -> Any:
    if name == "lightgbm":
        if LGBMRegressor is None:
            raise RuntimeError("LightGBM is installed incorrectly in this environment; skipping lightgbm.")
        return LGBMRegressor(
            objective="regression",
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
            **params,
        )
    if name == "xgboost":
        return XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
            verbosity=0,
            **params,
        )
    if name == "hist_gbdt":
        return HistGradientBoostingRegressor(random_state=SEED, loss="squared_error", **params)
    if name == "extra_trees":
        return ExtraTreesRegressor(random_state=SEED, n_jobs=-1, bootstrap=False, **params)
    if name == "random_forest":
        return RandomForestRegressor(random_state=SEED, n_jobs=-1, bootstrap=True, **params)
    if name == "ridge":
        return Ridge(random_state=SEED, **params)
    raise ValueError(f"Unknown model: {name}")


def suggest_params(trial: optuna.Trial, name: str) -> dict[str, Any]:
    if name == "lightgbm":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 300, 1800),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.09, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 24, 180),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 180),
            "subsample": trial.suggest_float("subsample", 0.65, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 30.0, log=True),
        }
    if name == "xgboost":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 180, 850),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 9),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 30.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.65, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 30.0, log=True),
        }
    if name == "hist_gbdt":
        return {
            "max_iter": trial.suggest_int("max_iter", 120, 650),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.10, log=True),
            "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 95),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 180),
            "l2_regularization": trial.suggest_float("l2_regularization", 1e-8, 20.0, log=True),
        }
    if name == "extra_trees":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 120, 500),
            "max_depth": trial.suggest_int("max_depth", 8, 40),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 25),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
            "max_features": trial.suggest_float("max_features", 0.35, 1.0),
        }
    if name == "random_forest":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 100, 420),
            "max_depth": trial.suggest_int("max_depth", 8, 35),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 25),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
            "max_features": trial.suggest_float("max_features", 0.35, 1.0),
        }
    if name == "ridge":
        return {
            "alpha": trial.suggest_float("alpha", 1e-3, 1000.0, log=True),
            "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        }
    raise ValueError(name)


def fit_predict(
    name: str,
    params: dict[str, Any],
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series | None = None,
) -> tuple[Any, np.ndarray]:
    model = get_model(name, params)
    if name == "lightgbm" and y_val is not None:
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            callbacks=[early_stopping(80, verbose=False), log_evaluation(period=0)],
        )
    else:
        model.fit(X_train, y_train)
    return model, model.predict(X_val)


def tune_model(
    name: str,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    n_trials: int,
) -> dict[str, Any]:
    def objective(trial: optuna.Trial) -> float:
        params = suggest_params(trial, name)
        _, pred = fit_predict(name, params, X_train, y_train, X_val, y_val)
        score = rmse(y_val.to_numpy(), clip_pred(pred))
        return score

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials, gc_after_trial=True, show_progress_bar=False)
    best_params = study.best_params
    best_model, val_pred = fit_predict(name, best_params, X_train, y_train, X_val, y_val)
    metrics = regression_metrics(y_val.to_numpy(), val_pred)
    return {
        "name": name,
        "best_params": best_params,
        "best_value": float(study.best_value),
        "metrics": metrics,
        "model": best_model,
        "val_pred": clip_pred(val_pred),
        "trials": study.trials_dataframe(),
    }


def optimize_blend(y_val: np.ndarray, preds: dict[str, np.ndarray], n_trials: int = 300) -> dict[str, Any]:
    names = list(preds)
    pred_matrix = np.vstack([preds[n] for n in names]).T

    def objective(trial: optuna.Trial) -> float:
        raw = np.array([trial.suggest_float(f"w_{name}", 0.0, 1.0) for name in names], dtype=float)
        if raw.sum() <= 0:
            raw = np.ones_like(raw)
        weights = raw / raw.sum()
        pred = clip_pred(pred_matrix @ weights)
        return rmse(y_val, pred)

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED + 7))
    study.optimize(objective, n_trials=n_trials, gc_after_trial=True, show_progress_bar=False)
    raw = np.array([study.best_params[f"w_{name}"] for name in names], dtype=float)
    weights = raw / raw.sum()
    pred = clip_pred(pred_matrix @ weights)
    return {
        "names": names,
        "weights": {name: float(weight) for name, weight in zip(names, weights)},
        "rmse": rmse(y_val, pred),
        "mae": float(mean_absolute_error(y_val, pred)),
        "r2": float(r2_score(y_val, pred)),
        "pred": pred,
    }


def save_feature_importance(results: list[dict[str, Any]], feature_columns: list[str]) -> None:
    rows = []
    for result in results:
        model = result["model"]
        if hasattr(model, "feature_importances_"):
            for feature, importance in zip(feature_columns, model.feature_importances_):
                rows.append({
                    "model": result["name"],
                    "feature": feature,
                    "importance": float(importance),
                })
    if rows:
        pd.DataFrame(rows).sort_values(["model", "importance"], ascending=[True, False]).to_csv(
            OUT / "feature_importance_validation_models.csv", index=False
        )


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--trials", type=int, default=12, help="Optuna trials per model")
    parser.add_argument(
        "--models",
        nargs="+",
        default=["lightgbm", "xgboost", "hist_gbdt", "extra_trees", "random_forest", "ridge"],
    )
    args = parser.parse_args()

    seed_everything()
    start = time.time()
    train, test = load_data()
    train_fit, valid = make_validation_split(train)
    y_fit = train_fit["demand"].astype(float)
    y_valid = valid["demand"].astype(float)

    builder = FeatureBuilder()
    X_fit, X_valid, artifacts = builder.build(train_fit.drop(columns=["demand"]), y_fit, valid.drop(columns=["demand"]))
    assert X_valid is not None

    print(f"Train rows for tuning: {len(X_fit):,}")
    print(f"Validation rows: {len(X_valid):,}")
    print(f"Feature count: {X_fit.shape[1]:,}")

    results: list[dict[str, Any]] = []
    for name in args.models:
        if name == "lightgbm" and LGBMRegressor is None:
            print("Skipping lightgbm: sklearn wrapper is not usable in this environment.")
            continue
        model_start = time.time()
        print(f"Tuning {name} with {args.trials} Optuna trials...")
        result = tune_model(name, X_fit, y_fit, X_valid, y_valid, args.trials)
        result["seconds"] = round(time.time() - model_start, 2)
        results.append(result)
        result["trials"].to_csv(OUT / f"optuna_trials_{name}.csv", index=False)
        print(
            f"{name}: RMSE={result['metrics']['rmse']:.6f}, "
            f"MAE={result['metrics']['mae']:.6f}, R2={result['metrics']['r2']:.6f}"
        )

    val_preds = {r["name"]: r["val_pred"] for r in results}
    blend = optimize_blend(y_valid.to_numpy(), val_preds, n_trials=max(250, args.trials * 40))

    summary_rows = []
    for result in results:
        summary_rows.append({
            "model": result["name"],
            **result["metrics"],
            "seconds": result["seconds"],
            "best_params_json": json.dumps(result["best_params"], sort_keys=True),
        })
    summary_rows.append({
        "model": "optuna_weighted_blend",
        "rmse": blend["rmse"],
        "mae": blend["mae"],
        "r2": blend["r2"],
        "seconds": 0.0,
        "best_params_json": json.dumps(blend["weights"], sort_keys=True),
    })
    summary = pd.DataFrame(summary_rows).sort_values("rmse")
    summary.to_csv(OUT / "validation_results.csv", index=False)

    valid_pred_df = valid[["Index", "geohash", "day", "timestamp", "demand"]].copy()
    for result in results:
        valid_pred_df[f"pred_{result['name']}"] = result["val_pred"]
    valid_pred_df["pred_blend"] = blend["pred"]
    valid_pred_df.to_csv(OUT / "validation_predictions.csv", index=False)

    best_single = min(results, key=lambda r: r["metrics"]["rmse"])
    save_feature_importance(results, artifacts.feature_columns)

    # Rebuild features on all available train.csv rows and predict test.csv.
    y_all = train["demand"].astype(float)
    X_all, X_test, final_artifacts = builder.build(train.drop(columns=["demand"]), y_all, test)
    assert X_test is not None

    final_models = {}
    test_predictions = {}
    for result in results:
        name = result["name"]
        model = get_model(name, result["best_params"])
        model.fit(X_all, y_all)
        pred = clip_pred(model.predict(X_test))
        final_models[name] = model
        test_predictions[name] = pred

        out = test[["Index"]].copy() if "Index" in test.columns else test[["geohash", "day", "timestamp"]].copy()
        out["demand"] = pred
        out.to_csv(OUT / f"submission_train_only_{name}.csv", index=False)

    blend_pred = np.zeros(len(test), dtype=float)
    for name, weight in blend["weights"].items():
        blend_pred += test_predictions[name] * weight
    blend_pred = clip_pred(blend_pred)
    blend_out = test[["Index"]].copy() if "Index" in test.columns else test[["geohash", "day", "timestamp"]].copy()
    blend_out["demand"] = blend_pred
    blend_out.to_csv(OUT / "submission_train_only_optuna_weighted_blend.csv", index=False)

    best_model_name = summary.iloc[0]["model"]
    best_pred_file = (
        "submission_train_only_optuna_weighted_blend.csv"
        if best_model_name == "optuna_weighted_blend"
        else f"submission_train_only_{best_model_name}.csv"
    )

    with (OUT / "best_params.json").open("w", encoding="utf-8") as f:
        json.dump(
            {
                "validation_strategy": "train on day48 plus day49 timestamps < 1:30; validate on day49 timestamps 1:30, 1:45, 2:00",
                "primary_metric": "RMSE, clipped predictions to [0, 1]",
                "best_model_or_blend": best_model_name,
                "best_prediction_file": best_pred_file,
                "models": {
                    r["name"]: {
                        "params": r["best_params"],
                        "metrics": r["metrics"],
                        "seconds": r["seconds"],
                    }
                    for r in results
                },
                "blend": {
                    "weights": blend["weights"],
                    "metrics": {"rmse": blend["rmse"], "mae": blend["mae"], "r2": blend["r2"]},
                },
                "feature_count": len(final_artifacts.feature_columns),
                "used_files": ["train.csv", "test.csv"],
                "explicitly_not_used": ["training.csv"],
                "elapsed_seconds": round(time.time() - start, 2),
            },
            f,
            indent=2,
        )

    joblib.dump(
        {
            "models": final_models,
            "blend_weights": blend["weights"],
            "feature_columns": final_artifacts.feature_columns,
            "best_model_or_blend": best_model_name,
            "best_prediction_file": best_pred_file,
        },
        OUT / "trained_models_train_only.joblib",
    )

    md = []
    md.append("# Train-Only Model Report\n")
    md.append("## Data policy\n")
    md.append("- Used only `train.csv` for labels/training and `test.csv` for prediction rows.")
    md.append("- Did not read or use `training.csv`.")
    md.append("\n## Validation strategy\n")
    md.append("- Fit/tune rows: day 48 plus day 49 timestamps before 1:30.")
    md.append("- Validation rows: day 49 timestamps 1:30, 1:45, and 2:00.")
    md.append(f"- Training rows for tuning: {len(X_fit):,}")
    md.append(f"- Validation rows: {len(X_valid):,}")
    md.append(f"- Engineered feature count: {X_fit.shape[1]:,}")
    md.append("\n## Results\n")
    for row in summary.to_dict(orient="records"):
        md.append(
            f"- {row['model']}: RMSE {row['rmse']:.6f}, MAE {row['mae']:.6f}, "
            f"R2 {row['r2']:.6f}"
        )
    md.append("\n## Best output\n")
    md.append(f"- Best validation choice: `{best_model_name}`")
    md.append(f"- Prediction file: `model_outputs/train_only/{best_pred_file}`")
    md.append("\n## Key files\n")
    md.append("- `model_outputs/train_only/validation_results.csv`")
    md.append("- `model_outputs/train_only/best_params.json`")
    md.append("- `model_outputs/train_only/validation_predictions.csv`")
    md.append("- `model_outputs/train_only/feature_importance_validation_models.csv`")
    md.append("- `model_outputs/train_only/trained_models_train_only.joblib`")
    (OUT / "train_only_model_report.md").write_text("\n".join(md) + "\n", encoding="utf-8")

    print("\nBest validation choice:", best_model_name)
    print("Best prediction file:", OUT / best_pred_file)
    print("Report:", OUT / "train_only_model_report.md")


# In this notebook, call main() explicitly from the run cell below.


In [ ]:
# Full tuned run used for the submitted predictions.
# You can reduce --trials for a quicker local smoke test.
import sys
sys.argv = [
    'train_models_train_only.py',
    '--trials', '6',
    '--models', 'lightgbm', 'xgboost', 'hist_gbdt', 'extra_trees', 'random_forest', 'ridge',
]
main()


In [ ]:
# Format the best model output exactly like sample_submission.csv.
import pandas as pd
from pathlib import Path

root = Path.cwd()
sample = pd.read_csv(root / 'sample_submission.csv')
test = pd.read_csv(root / 'test.csv')
best = pd.read_csv(root / 'model_outputs' / 'train_only' / 'submission_train_only_optuna_weighted_blend.csv')

predictions = best[list(sample.columns)].copy()
predictions['demand'] = pd.to_numeric(predictions['demand'], errors='coerce').clip(0.0, 1.0)
assert list(predictions.columns) == list(sample.columns)
assert len(predictions) == len(test)
assert predictions['Index'].equals(test['Index'])
assert predictions['demand'].notna().all()

predictions.to_csv('predictions.csv', index=False)
predictions.head()
